# v108_stage2_more_data — stage 2 on fit + tune entities, 127 leaves

| Field | Value |
|---|---|
| **Version** | `v108_stage2_more_data` |
| **Plan group** | D3 / E2 (stage-2 matcher, decision layer) |
| **Parent version** | v106 |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

A stage-2 variant of v106: the same mock fold, stage 1, candidate filter and stage-1 outputs
(read back from v106's stage-1 cache, so no stage-1 pass runs); only stage 2 and the rule
are refitted. Rules are tuned on the tight mock and versions compared by est_public.

## 1. Hypothesis

* **Change vs parent (v106):** stage 2 trained on the mock's fit AND tune entities (both scored out of fold; early stopping on each model's held-out part), 127 leaves
* **Why:** stage 2 has half as much data again and more capacity; below-threshold misses (1.96 % of true pairs) come from pairs stage 2 cannot yet separate
* **Discard if:** est_public does not beat v106 by more than 0.001.

## 2. Setup

The parent's exact configuration comes back from its `config.json` (`PipelineConfig.from_record`)
and its two-stage artifacts (`TwoStage.load`); this version's stage-2 parameters follow.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.data import isin
from entity_resolution.decision import apply_rule, tune_expected
from entity_resolution.evaluate import error_samples
from entity_resolution.mock import FP_WEIGHT, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import PipelineConfig, mock_scores, peak_rss_gb, tune_mock
from entity_resolution.split import load_fold
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    TwoStage, fit_stage2, mock_scored, mock_stage1, predict_stage2, run_test_two_stage,
)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v108_stage2_more_data"
ARTIFACTS = EXP_DIR / "artifacts"
PARENT_DIR = next(C.EXPERIMENTS.glob("v106_*"))
cfg = PipelineConfig.from_record(json.loads((PARENT_DIR / "artifacts" / "stage1" /
                                            "config.json").read_text()))
parent_ts = TwoStage.load(PARENT_DIR / "artifacts", cfg)
parent = json.loads((PARENT_DIR / "metrics.json").read_text())
STAGE1_CACHE = Path(parent["metrics"]["stage1_cache"])
SEEDS = [42]
DROP = []
params = MatcherParams(backend="xgb", device="cuda", num_leaves=127,
                       learning_rate=0.05, n_estimators=4000,
                       min_child_weight=1.0)
tcfg = replace(parent_ts.tcfg, model=params,
              train_roles=tuple(['fit', 'tune']))
EST_TO_BEAT = parent["metrics"]["est_public"]
timings: dict[str, float] = {}
t_start = time.time()
print("parent", PARENT_DIR.name, "est_public", round(EST_TO_BEAT, 4), "| blocking",
      cfg.blocking.key(), "| stage-1 cache", STAGE1_CACHE.name)
print(json.dumps(asdict(params), indent=1))

## 3. Data

The mock fold and the parent's cached stage-1 outputs of every present entity.

In [ ]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID],
                      sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID], target_shape())
    del train, val, fit_fold, tune_fold
    outs = mock_stage1(cfg, parent_ts.stage1, mock, parent_ts.tcfg,
                       cache_dir=STAGE1_CACHE / "mock")
cols = [c for c in next(iter(outs.values())).X.columns if c not in DROP]
{c: len(o.pairs) for c, o in outs.items()}, len(cols)

## 4. Method

Stage 2 refitted with this version's parameters (one cross-fitted pair of models per seed;
several seeds are averaged), the 1-to-1 across every present entity, then the threshold grid
and expected-F0.5 decoding, both tuned for the tight score; the better one is the rule.

In [ ]:
t0 = time.time()
models = []
for seed in SEEDS:
    m, fit_info = fit_stage2(outs, mock, replace(tcfg, model=replace(params, seed=seed)),
                             columns=cols)
    models += m
timings["fit_seconds"] = round(time.time() - t0, 2)
# with several seeds, fold f's models are models[f::folds]; average them per fold
folds = tcfg.folds
per_fold = [models[f::folds] for f in range(folds)]


class Mean:
    """The mean probability of several models (one cross-fitting part, several seeds)."""

    def __init__(self, ms):
        self.ms, self.feature_names_ = ms, ms[0].feature_names_

    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.ms], axis=0).astype(np.float32)

    def save(self, out):
        for i, m in enumerate(self.ms):
            m.save(Path(out) / f"seed{i}")
        return out


stage2 = [Mean(ms) if len(ms) > 1 else ms[0] for ms in per_fold]
scored, report = mock_scored(outs, stage2, mock, tcfg)
rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
tp = mock.part("tune")
rows_t = scored[isin(scored[C.S1_ID], pd.Index(tp.s1[C.ENTITY_ID]))]
rule_e, table_e = tune_expected(rows_t, tp.s1[C.ENTITY_ID], tp.pairs,
                                gammas=(0.7, 0.85, 1.0, 1.2, 1.5, 2.0),
                                misses=(0.0, 0.05, 0.1, 0.2, 0.4), fp_weight=FP_WEIGHT)
rule, table = ((rule_e, table_e) if table_e["f_beta"].max() > table_t["f_beta"].max()
               else (rule_t, table_t))
res = mock_scores(scored, mock, rule)
timings["tune_seconds"] = round(time.time() - t0 - timings["fit_seconds"], 2)
print(f"fit {timings['fit_seconds']:.0f} s; rule {rule}")
res[["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision", "pair_recall",
     "entities"]].round(4)

## 5. Evaluation

In [ ]:
cmp = pd.DataFrame({PARENT_DIR.name: {"est_public": EST_TO_BEAT,
                                      "mock_f05": float(parent["mock_f05"])},
                    EXP_DIR.name: {"est_public": res.loc["all", "est_public"],
                                   "mock_f05": res.loc["all", "f_beta"]}}).T
cmp["delta"] = cmp["est_public"] - EST_TO_BEAT
cmp.round(4)

## 6. Error analysis

In [ ]:
part = mock.part("val")
matches = apply_rule(scored[isin(scored[C.S1_ID], pd.Index(part.s1[C.ENTITY_ID]))], rule)
counts = {k: len(error_samples(matches, part, k, n=10**9))
          for k in ("false_merge", "missed", "false_singleton", "singleton_merge")}
counts

## 7. Log the result

In [ ]:
ts = TwoStage(parent_ts.stage1, stage2, rule, tcfg, table, {"seeds": SEEDS, "drop": DROP})
if len(SEEDS) == 1:                   # plain Matchers: the version is saved and reloadable
    ts.save(ARTIFACTS)
record = {
    "hypothesis": "stage-2 variant; see section 1", "stage2_params": asdict(params),
    "seeds": SEEDS, "drop_columns": DROP, "rule": asdict(rule), "rule_kind": type(rule).__name__,
    "mock_f_beta": res.loc["all", "f_beta"], "est_public": res.loc["all", "est_public"],
    "f_tight": res.loc["all", "f_tight"], "stage1_cache": str(STAGE1_CACHE),
    "by_country": res.to_dict("index"), "errors_mock": counts, **timings,
}
DECISION = "KEEP" if record["est_public"] > EST_TO_BEAT + 0.001 else "DROP"
record["decision"] = DECISION
print(f"est_public {EST_TO_BEAT:.4f} -> {record['est_public']:.4f} {DECISION}")
row = log_result(EXP_DIR, change="stage 2 trained on the mock's fit AND tune entities (both scored out of fold; early stopping on each model's held-out pa", group="D3", mock_f05=record["mock_f_beta"],
                 notes=f"est_public {record['est_public']:.4f}", metrics=record, owner="M1",
                 parent=PARENT_DIR.name.split("_")[0], decision=DECISION)
row

## 8. Conclusion

Written after the run from the numbers above.

## 9. Test inference

The parent's cached stage-1 test outputs, this version's stage 2 and rule; files in
`submissions/<version>/`, then both validators.

In [ ]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
    cfg, ts, cache_dir=STAGE1_CACHE / "test")
print(f"run_test {time.time() - t0:.0f} s")
dest = C.ROOT / "submissions" / "v108"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")